In [1]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
# Load the Kaggle competition files.
# Keeping these paths centralized makes it easy to rerun in Kaggle without touching later cells.
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e5"

df = pd.read_csv(f"{DATA_DIR}/train.csv")
TEST_PATH = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import time


# Drop id
df = df.drop(columns=['id'])

# Prepare categorical columns
cat_cols = ['Driver', 'Compound', 'Race']
for col in cat_cols:
    df[col] = df[col].astype('category')

X = df.drop(columns=['PitNextLap'])
y = df['PitNextLap']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

# Initialize and train XGBoost
print("Training XGBoost baseline...")
start_time = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1
)

clf.fit(X_train, y_train)
elapsed = time.time() - start_time
print(f"Training completed in {elapsed:.2f} seconds.")

# Evaluate
y_train_pred = clf.predict_proba(X_train)[:, 1]
y_val_pred = clf.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred)
val_auc = roc_auc_score(y_val, y_val_pred)

print(f"Train ROC-AUC: {train_auc:.5f}")
print(f"Validation ROC-AUC: {val_auc:.5f}")

X_train shape: (351312, 14), X_val shape: (87828, 14)
Training XGBoost baseline...
Training completed in 7.59 seconds.
Train ROC-AUC: 0.97477
Validation ROC-AUC: 0.93986


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Data loaded. Train shape:", train_df.shape, "Test shape:", test_df.shape)
# Drop id from features but keep for output
train_ids = train_df['id']
test_ids = test_df['id']
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])
# Feature engineering
print("Engineering features...")
for df in [train_df, test_df]:
    df['Estimated_Total_Laps'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-8)).replace([np.inf, -np.inf], np.nan)
    df['Estimated_Total_Laps'] = df['Estimated_Total_Laps'].fillna(70).round().clip(10, 100)
    df['Remaining_Laps'] = df['Estimated_Total_Laps'] - df['LapNumber']
    df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-8)
    df['TyreLife_x_Degradation'] = df['TyreLife'] * df['Cumulative_Degradation']
# Median pit tyre life
pit_stops = train_df[train_df['PitNextLap'] == 1]
median_pit = pit_stops.groupby(['Race', 'Compound', 'Stint'])['TyreLife'].median().reset_index()
median_pit.rename(columns={'TyreLife': 'MedianPitTyreLife'}, inplace=True)
global_median = pit_stops['TyreLife'].median()
comp_median = pit_stops.groupby('Compound')['TyreLife'].median().to_dict()
train_df = train_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
train_df['MedianPitTyreLife'] = train_df['MedianPitTyreLife'].fillna(train_df['Compound'].map(comp_median)).fillna(global_median)
train_df['TyreLife_to_MedianPitRatio'] = train_df['TyreLife'] / (train_df['MedianPitTyreLife'] + 1e-8)
test_df = test_df.merge(median_pit, on=['Race','Compound','Stint'], how='left')
test_df['MedianPitTyreLife'] = test_df['MedianPitTyreLife'].fillna(test_df['Compound'].map(comp_median)).fillna(global_median)
test_df['TyreLife_to_MedianPitRatio'] = test_df['TyreLife'] / (test_df['MedianPitTyreLife'] + 1e-8)
# Median lap time
median_lap = train_df.groupby(['Race','Compound'])['LapTime (s)'].median().reset_index()
median_lap.rename(columns={'LapTime (s)': 'MedianLapTimeRC'}, inplace=True)
global_lap = train_df['LapTime (s)'].median()
train_df = train_df.merge(median_lap, on=['Race','Compound'], how='left')
train_df['MedianLapTimeRC'] = train_df['MedianLapTimeRC'].fillna(global_lap)
train_df['LapTime_Ratio'] = train_df['LapTime (s)'] / (train_df['MedianLapTimeRC'] + 1e-8)
test_df = test_df.merge(median_lap, on=['Race','Compound'], how='left')
test_df['MedianLapTimeRC'] = test_df['MedianLapTimeRC'].fillna(global_lap)
test_df['LapTime_Ratio'] = test_df['LapTime (s)'] / (test_df['MedianLapTimeRC'] + 1e-8)
# Categorical conversion
cat_cols = ['Driver','Compound','Race']
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')
# Prepare features and target
y_train = train_df['PitNextLap']
X_train = train_df.drop(columns=['PitNextLap'])
X_test = test_df.copy()
print('Feature matrix shapes:', X_train.shape, X_test.shape)
# XGBoost model
clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    min_child_weight=3,
    random_state=42,
    enable_categorical=True,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='auc'
)
print('Training final model on full data...')
start = time.time()
clf.fit(X_train, y_train)
print(f'Training completed in {time.time() - start:.2f} seconds.')
print('Predicting on test set...')
test_pred = clf.predict_proba(X_test)[:,1]
pred_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_pred})


Data loaded. Train shape: (439140, 16) Test shape: (188165, 15)
Engineering features...
Feature matrix shapes: (439140, 22) (188165, 22)
Training final model on full data...
Training completed in 47.41 seconds.
Predicting on test set...


In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD DATA
# ==========================================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Train:", train_df.shape, "| Test:", test_df.shape)

train_ids = train_df['id']
test_ids  = test_df['id']
y_train   = train_df['PitNextLap']

# Race-level groups for strict GroupKFold (prevents data leakage across races)
train_df['Race_Group'] = train_df['Year'].astype(str) + "_" + train_df['Race'].astype(str)
groups = train_df['Race_Group']

# ==========================================
# 2. FEATURE ENGINEERING v5 — ELITE PACK
# ==========================================
def engineer_features_v5(df, is_train=True, global_stats=None):
    d = df.copy()
    d = d.sort_values(['Year', 'Race', 'Driver', 'LapNumber']).reset_index(drop=True)
    G  = ['Year', 'Race', 'Driver']   # driver-level group
    GL = ['Year', 'Race', 'LapNumber'] # race-lap group

    # ── A. RACE STRUCTURE ──────────────────────────────────────────────────────
    d['Est_Total_Laps']   = (d['LapNumber'] / (d['RaceProgress'] + 1e-8)) \
                             .replace([np.inf, -np.inf], np.nan).fillna(70).round().clip(10, 100)
    d['Remaining_Laps']   = d['Est_Total_Laps'] - d['LapNumber']
    d['Remaining_Laps_Sq']= d['Remaining_Laps'] ** 2
    d['Race_Phase']       = pd.cut(d['RaceProgress'], bins=[0,.33,.66,1.01],
                                    labels=[0,1,2], right=False).astype(float)
    d['Is_Early_Race']    = (d['RaceProgress'] < 0.30).astype(int)
    d['Is_Late_Race']     = (d['RaceProgress'] > 0.70).astype(int)

    # ── B. COMPOUND-SPECIFIC TYRE PHYSICS ────────────────────────────────────
    # Real F1 expected stint lengths per compound
    compound_life = {'SOFT': 20, 'MEDIUM': 30, 'HARD': 45,
                     'INTERMEDIATE': 25, 'WET': 20}
    d['Expected_Compound_Life'] = d['Compound'].map(compound_life).fillna(30)
    d['TyreLife_vs_Expected']   = d['TyreLife'] / (d['Expected_Compound_Life'] + 1e-8)
    d['TyreLife_Overrun']       = (d['TyreLife'] - d['Expected_Compound_Life']).clip(lower=0)
    d['In_Pit_Window']          = ((d['TyreLife_vs_Expected'] >= 0.75) &
                                    (d['TyreLife_vs_Expected'] <= 1.30)).astype(int)

    # Polynomial tyre life features
    d['TyreLife_Sq']   = d['TyreLife'] ** 2
    d['TyreLife_Cube'] = d['TyreLife'] ** 3

    # Degradation physics
    d['Degradation_Rate']  = d['Cumulative_Degradation'] / (d['TyreLife'] + 1e-8)
    d['Deg_Accel']         = d.groupby(G)['Degradation_Rate'].diff()   # 2nd derivative
    d['TyreLife_x_Deg']    = d['TyreLife'] * d['Cumulative_Degradation']
    d['Critical_Deg_Zone'] = (d['Cumulative_Degradation'] > 15).astype(int)
    d['Severe_Deg_Zone']   = (d['Cumulative_Degradation'] > 25).astype(int)

    # ── C. STINT DETECTION ────────────────────────────────────────────────────
    d['TyreLife_Diff'] = d.groupby(G)['TyreLife'].diff()
    d['Stint_Start']   = ((d['TyreLife_Diff'] < 0) | (d['TyreLife'] == 1)).astype(int)
    d['Stint_Number']  = d.groupby(G)['Stint_Start'].cumsum()
    d['Is_First_Stint']= (d['Stint_Number'] == 1).astype(int)
    d['Is_3rd_Stint+'] = (d['Stint_Number'] >= 3).astype(int)

    # ── D. LAG FEATURES (micro-level time series) ─────────────────────────────
    for lag in [1, 2, 3, 5]:
        d[f'LapTime_Lag_{lag}']  = d.groupby(G)['LapTime (s)'].shift(lag)
        d[f'Deg_Lag_{lag}']      = d.groupby(G)['Cumulative_Degradation'].shift(lag)
    for lag in [1, 2, 3]:
        d[f'Pos_Lag_{lag}']      = d.groupby(G)['Position'].shift(lag)

    # ── E. MOMENTUM & DELTA ───────────────────────────────────────────────────
    d['LapTime_Delta_1'] = d['LapTime (s)'] - d['LapTime_Lag_1']
    d['LapTime_Delta_2'] = d['LapTime_Lag_1'] - d['LapTime_Lag_2']
    d['LapTime_Accel']   = d['LapTime_Delta_1'] - d['LapTime_Delta_2']   # jerk
    d['Pos_Delta_1']     = d['Position'] - d['Pos_Lag_1']
    d['Pos_Delta_3']     = d['Position'] - d['Pos_Lag_3']

    # ── F. ROLLING STATISTICS ────────────────────────────────────────────────
    for w in [3, 5, 8]:
        d[f'LT_RollMean_{w}'] = d.groupby(G)['LapTime (s)'].transform(
            lambda x: x.rolling(w, min_periods=1).mean())
        d[f'LT_RollStd_{w}']  = d.groupby(G)['LapTime (s)'].transform(
            lambda x: x.rolling(w, min_periods=1).std()).fillna(0)
        d[f'Deg_RollMean_{w}']= d.groupby(G)['Cumulative_Degradation'].transform(
            lambda x: x.rolling(w, min_periods=1).mean())

    # Rolling min = best recent pace (pace drop relative to this = degradation signal)
    d['LT_RollMin_5']    = d.groupby(G)['LapTime (s)'].transform(
        lambda x: x.rolling(5, min_periods=1).min())
    d['LT_vs_RollMin']   = d['LapTime (s)'] - d['LT_RollMin_5']

    # Pace anomaly (sudden slowdown = pitting next lap)
    d['Pace_Drop_3']     = d['LapTime (s)'] - d['LT_RollMean_3']
    d['Pace_Drop_5']     = d['LapTime (s)'] - d['LT_RollMean_5']

    # Linear trend slope (using polyfit over 5-lap window)
    def rolling_slope(series, window=5):
        result = series.copy() * 0.0
        arr = series.values
        for i in range(len(arr)):
            start = max(0, i - window + 1)
            segment = arr[start:i+1]
            if len(segment) >= 3:
                result.iloc[i] = np.polyfit(range(len(segment)), segment, 1)[0]
        return result

    d['LapTime_Trend']   = d.groupby(G)['LapTime (s)'].transform(
        lambda x: rolling_slope(x, 5))
    d['Deg_Trend']       = d.groupby(G)['Cumulative_Degradation'].transform(
        lambda x: rolling_slope(x, 5))

    # ── G. RACE-WIDE CONTEXT ─────────────────────────────────────────────────
    # How is driver doing vs the whole field this lap?
    d['Race_Avg_LapTime']  = d.groupby(GL)['LapTime (s)'].transform('mean')
    d['Relative_LapTime']  = d['LapTime (s)'] - d['Race_Avg_LapTime']
    d['Race_Avg_Deg']      = d.groupby(GL)['Cumulative_Degradation'].transform('mean')
    d['Relative_Deg']      = d['Cumulative_Degradation'] - d['Race_Avg_Deg']
    d['Field_Avg_TyreLife']= d.groupby(GL)['TyreLife'].transform('mean')
    d['Relative_TyreLife'] = d['TyreLife'] - d['Field_Avg_TyreLife']

    # ── H. STRATEGIC PIT WINDOW MAPPING ──────────────────────────────────────
    if is_train:
        pits = d[d['PitNextLap'] == 1]

        # Median/Std pit TyreLife by Race + Compound
        med_pit = pits.groupby(['Race','Compound'])['TyreLife'] \
                      .agg(['median','std']).reset_index() \
                      .rename(columns={'median':'MedPitLife','std':'StdPitLife'})

        # Compound-level overall median
        comp_pit = pits.groupby('Compound')['TyreLife'].median().reset_index() \
                       .rename(columns={'TyreLife':'Comp_MedPitLife'})

        # Driver aggressiveness (how early/late they typically pit)
        drv_pit = pits.groupby('Driver')['TyreLife'].median().reset_index() \
                      .rename(columns={'TyreLife':'Drv_AvgPitLife'})

        global_med_pit  = pits['TyreLife'].median()
        global_med_lap  = d['LapTime (s)'].median()
        global_stats    = (med_pit, comp_pit, drv_pit, global_med_pit, global_med_lap)
    else:
        med_pit, comp_pit, drv_pit, global_med_pit, global_med_lap = global_stats

    d = d.merge(med_pit, on=['Race','Compound'], how='left')
    d['MedPitLife']  = d['MedPitLife'].fillna(global_med_pit)
    d['StdPitLife']  = d['StdPitLife'].fillna(5.0)
    d['TyreLife_Remaining_Window'] = d['MedPitLife'] - d['TyreLife']
    d['TyreLife_Ratio']            = d['TyreLife'] / (d['MedPitLife'] + 1e-8)
    d['TyreLife_Zscore']           = (d['TyreLife'] - d['MedPitLife']) / (d['StdPitLife'] + 1e-8)

    d = d.merge(comp_pit, on='Compound', how='left')
    d['Comp_MedPitLife']          = d['Comp_MedPitLife'].fillna(global_med_pit)
    d['TyreLife_vs_CompMed']      = d['TyreLife'] / (d['Comp_MedPitLife'] + 1e-8)

    d = d.merge(drv_pit, on='Driver', how='left')
    d['Drv_AvgPitLife']           = d['Drv_AvgPitLife'].fillna(global_med_pit)
    d['Drv_TyreLife_Ratio']       = d['TyreLife'] / (d['Drv_AvgPitLife'] + 1e-8)

    # ── I. INTERACTION FEATURES ───────────────────────────────────────────────
    d['TyreLife_x_RaceProgress']  = d['TyreLife']  * d['RaceProgress']
    d['Deg_x_Remaining']          = d['Cumulative_Degradation'] * d['Remaining_Laps']
    d['TyreLife_x_Remaining']     = d['TyreLife']  * d['Remaining_Laps']
    d['Deg_x_Phase']              = d['Cumulative_Degradation'] * d['Race_Phase']
    d['Remaining_vs_TyreLife']    = d['Remaining_Laps'] / (d['TyreLife'] + 1e-8)  # must pit urgency

    # ── J. COMPOSITE PIT URGENCY SCORE ───────────────────────────────────────
    # Combines multiple signals into one engineered feature
    d['Pit_Urgency'] = (
        d['TyreLife_Ratio'].clip(0, 3)          * 0.35 +
        d['TyreLife_vs_Expected'].clip(0, 3)    * 0.30 +
        (d['TyreLife_Zscore'].clip(-4, 4) + 4)  * 0.20 +   # normalize to positive
        d['Pace_Drop_5'].clip(-5, 20) / 25      * 0.15
    )

    # ── K. CLEAN UP ───────────────────────────────────────────────────────────
    drop_internal = ['TyreLife_Diff', 'Stint_Start']
    d = d.drop(columns=[c for c in drop_internal if c in d.columns])
    d = d.fillna(0)

    for col in ['Driver', 'Compound', 'Race']:
        d[col] = d[col].astype('category')

    if is_train:
        return d, global_stats
    return d

# ── Build features ────────────────────────────────────────────────────────────
print("\nEngineering Features v5 (Elite)...")
X_train_feat, stats = engineer_features_v5(train_df, is_train=True)
X_test_feat          = engineer_features_v5(test_df,  is_train=False, global_stats=stats)

# Re-align to original row order
X_train_feat = X_train_feat.set_index('id').loc[train_ids].reset_index()
X_test_feat  = X_test_feat.set_index('id').loc[test_ids].reset_index()

DROP_TRAIN = ['id', 'PitNextLap', 'Race_Group']
DROP_TEST  = ['id']
X_train = X_train_feat.drop(columns=[c for c in DROP_TRAIN if c in X_train_feat.columns])
X_test  = X_test_feat.drop(columns=[c for c in DROP_TEST  if c in X_test_feat.columns])

print(f"Features ready — Train: {X_train.shape} | Test: {X_test.shape}")

# ==========================================
# 3. GROUP K-FOLD + 4-MODEL ENSEMBLE
# ==========================================
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

CAT_COLS = ['Driver', 'Compound', 'Race']
cat_idx  = [X_train.columns.get_loc(c) for c in CAT_COLS]

# Accumulators
oof_xgb  = np.zeros(len(X_train)); test_xgb  = np.zeros(len(X_test))
oof_lgb  = np.zeros(len(X_train)); test_lgb  = np.zeros(len(X_test))
oof_cat  = np.zeros(len(X_train)); test_cat  = np.zeros(len(X_test))
oof_hgb  = np.zeros(len(X_train)); test_hgb  = np.zeros(len(X_test))

print(f"\nRunning {N_SPLITS}-Fold Group Cross-Validation...\n")

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_va, y_va = X_train.iloc[va_idx], y_train.iloc[va_idx]

    # ─── MODEL 1: XGBoost ─────────────────────────────────────────────────
    m_xgb = xgb.XGBClassifier(
        n_estimators=5000, max_depth=6, learning_rate=0.008,
        subsample=0.80, colsample_bytree=0.80, colsample_bylevel=0.80,
        reg_alpha=4.0, reg_lambda=8.0, min_child_weight=20,
        gamma=0.05, grow_policy='lossguide', max_leaves=63,
        random_state=42 + fold, enable_categorical=True,
        tree_method='hist', n_jobs=-1, eval_metric='auc',
        early_stopping_rounds=150
    )
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    # ─── MODEL 2: LightGBM ────────────────────────────────────────────────
    m_lgb = LGBMClassifier(
        n_estimators=5000, num_leaves=63, max_depth=-1,
        learning_rate=0.008, subsample=0.80, colsample_bytree=0.80,
        reg_alpha=4.0, reg_lambda=8.0, min_child_samples=20,
        random_state=42 + fold, n_jobs=-1, verbose=-1, boosting_type='gbdt'
    )
    m_lgb.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
        callbacks=[early_stopping(stopping_rounds=150, verbose=False),
                   log_evaluation(period=-1)]
    )

    # ─── MODEL 3: CatBoost ────────────────────────────────────────────────
    X_tr_cb  = X_tr.copy(); X_va_cb  = X_va.copy(); X_te_cb  = X_test.copy()
    for c in CAT_COLS:
        X_tr_cb[c]  = X_tr_cb[c].astype(str)
        X_va_cb[c]  = X_va_cb[c].astype(str)
        X_te_cb[c]  = X_te_cb[c].astype(str)

    m_cat = CatBoostClassifier(
        iterations=5000, depth=6, learning_rate=0.015,
        l2_leaf_reg=8, bagging_temperature=0.5, random_strength=1.5,
        border_count=128, random_seed=42 + fold, eval_metric='AUC',
        early_stopping_rounds=150, task_type='CPU', thread_count=-1, verbose=False
    )
    m_cat.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), cat_features=cat_idx)

    # ─── MODEL 4: HistGradientBoosting (sklearn, great with mixed features) ──
    X_tr_hgb = X_tr.copy(); X_va_hgb = X_va.copy(); X_te_hgb = X_test.copy()
    for c in CAT_COLS:
        X_tr_hgb[c] = X_tr_hgb[c].cat.codes.astype(float)
        X_va_hgb[c] = X_va_hgb[c].cat.codes.astype(float)
        X_te_hgb[c] = X_te_hgb[c].cat.codes.astype(float)

    m_hgb = HistGradientBoostingClassifier(
        max_iter=2000, max_leaf_nodes=63, learning_rate=0.01,
        min_samples_leaf=20, l2_regularization=8.0,
        random_state=42 + fold
    )
    m_hgb.fit(X_tr_hgb, y_tr)

    # ─── OOF predictions ──────────────────────────────────────────────────
    oof_xgb[va_idx] = m_xgb.predict_proba(X_va)[:, 1]
    oof_lgb[va_idx] = m_lgb.predict_proba(X_va)[:, 1]
    oof_cat[va_idx] = m_cat.predict_proba(X_va_cb)[:, 1]
    oof_hgb[va_idx] = m_hgb.predict_proba(X_va_hgb)[:, 1]

    s = {k: roc_auc_score(y_va, v)
         for k, v in [('XGB', oof_xgb[va_idx]), ('LGB', oof_lgb[va_idx]),
                      ('CAT', oof_cat[va_idx]), ('HGB', oof_hgb[va_idx])]}
    print(f"Fold {fold+1} | " + " | ".join(f"{k}: {v:.5f}" for k, v in s.items()))

    # ─── Test predictions (averaged across folds) ─────────────────────────
    test_xgb  += m_xgb.predict_proba(X_test)[:, 1]   / N_SPLITS
    test_lgb  += m_lgb.predict_proba(X_test)[:, 1]   / N_SPLITS
    test_cat  += m_cat.predict_proba(X_te_cb)[:, 1]  / N_SPLITS
    test_hgb  += m_hgb.predict_proba(X_te_hgb)[:, 1] / N_SPLITS

# ==========================================
# 4. OOF SUMMARY & OPTIMIZED BLENDING
# ==========================================
print("\n═══════════════ OOF SCORES ═══════════════")
oof_matrix  = np.column_stack([oof_xgb, oof_lgb, oof_cat, oof_hgb])
test_matrix = np.column_stack([test_xgb, test_lgb, test_cat, test_hgb])
names       = ['XGB', 'LGB', 'CatBoost', 'HGB']

for n, o in zip(names, [oof_xgb, oof_lgb, oof_cat, oof_hgb]):
    print(f"  OOF {n:10s}: {roc_auc_score(y_train, o):.5f}")

# ── Optimize ensemble weights via Nelder-Mead on OOF ──────────────────────────
def neg_auc(w):
    w = np.abs(w) / (np.abs(w).sum() + 1e-12)
    return -roc_auc_score(y_train, (oof_matrix * w).sum(axis=1))

res = minimize(neg_auc, x0=np.ones(4)/4, method='Nelder-Mead',
               options={'maxiter': 20000, 'xatol': 1e-10, 'fatol': 1e-10})
opt_w = np.abs(res.x) / np.abs(res.x).sum()
print(f"\nOptimal weights: " + ", ".join(f"{n}={w:.3f}" for n, w in zip(names, opt_w)))

# ── Rank-average blending (more robust than raw probability blend) ─────────────
def rank_blend(mat, weights):
    ranked = np.column_stack([rankdata(mat[:, i]) / mat.shape[0]
                              for i in range(mat.shape[1])])
    return (ranked * weights).sum(axis=1)

blend_prob = (test_matrix * opt_w).sum(axis=1)         # weighted probability blend
blend_rank = rank_blend(test_matrix, opt_w)             # weighted rank blend
final_preds = 0.55 * blend_prob + 0.45 * blend_rank    # mix both

oof_blend_prob = (oof_matrix * opt_w).sum(axis=1)
print(f"\nFinal Ensemble OOF AUC: {roc_auc_score(y_train, oof_blend_prob):.5f}")

# ==========================================
# 5. SUBMISSION
# ==========================================
OUTPUT_PATH = "submission.csv"
pd.DataFrame({'id': test_ids, 'PitNextLap': final_preds}).to_csv(OUTPUT_PATH, index=False)
print(f"\n✅  Submission saved → {OUTPUT_PATH}")

Train: (439140, 16) | Test: (188165, 15)

Engineering Features v5 (Elite)...
Features ready — Train: (439140, 86) | Test: (188165, 86)

Running 5-Fold Group Cross-Validation...

Fold 1 | XGB: 0.92402 | LGB: 0.91948 | CAT: 0.92662 | HGB: 0.92007
Fold 2 | XGB: 0.88718 | LGB: 0.88159 | CAT: 0.90232 | HGB: 0.84628
Fold 3 | XGB: 0.93617 | LGB: 0.93281 | CAT: 0.93376 | HGB: 0.93949
Fold 4 | XGB: 0.92605 | LGB: 0.92745 | CAT: 0.93032 | HGB: 0.93114
Fold 5 | XGB: 0.91294 | LGB: 0.91091 | CAT: 0.89547 | HGB: 0.92050

═══════════════ OOF SCORES ═══════════════
  OOF XGB       : 0.91722
  OOF LGB       : 0.90482
  OOF CatBoost  : 0.88682
  OOF HGB       : 0.90973

Optimal weights: XGB=0.717, LGB=0.000, CatBoost=0.048, HGB=0.235

Final Ensemble OOF AUC: 0.91804

✅  Submission saved → submission.csv
